# Day 5 — [프로젝트 1] 정형 데이터 예측 서비스

## 1. 프로젝트 개요: 정형 데이터 기반 예측 서비스 설계

In [1]:
# 서버 실행 도우미 — 노트북 맨 처음에 한 번 실행하세요.
# 노트북 안에서 uvicorn 서버를 띄우고 멈추는 함수를 정의합니다.
import os, sys, asyncio, threading, time, socket, contextlib
import uvicorn

# 작업 디렉터리를 app/ 가 있는 위치로 맞춥니다 (notebooks/ 안에서 열어도 동작).
if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# 코드를 저장할 폴더를 미리 만들어 둡니다.
for _d in ('app', 'models', 'data', 'frontend'):
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}  # port -> (server, thread)

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """실행 중인 서버를 멈춥니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='warning'):
    """백그라운드에서 uvicorn 서버를 띄웁니다.

    app: FastAPI 객체 또는 'app.main:app' 같은 import 경로.
    같은 포트에 서버가 이미 있으면 먼저 멈추고 새로 띄웁니다.
    """
    stop_server(port)
    if _port_open(host, port):
        print(f'⚠️ 포트 {port}를 다른 프로세스가 사용 중입니다 (다른 노트북의 서버일 가능성).')
        print('   그 노트북에서 stop_server(8000)을 실행하거나 커널을 종료한 뒤, 이 셀을 다시 실행하세요.')
        return None
    if isinstance(app, str):
        sys.modules.pop(app.split(':')[0], None)   # 파일을 다시 저장한 경우 최신 내용 반영
    for _ in range(50):
        if not _port_open(host, port):
            break
        time.sleep(0.1)
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        # Windows는 SelectorEventLoop, 그 외는 기본 이벤트 루프를 사용합니다.
        if sys.platform == 'win32':
            loop = asyncio.SelectorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    for _ in range(40):
        if _port_open(host, port):
            print(f'서버 실행됨: http://{host}:{port}')
            return server
        time.sleep(0.25)
    print('서버가 시작되지 않았습니다. 위 로그를 확인하세요.')
    return server

print('서버 도우미 준비 완료 (serve_in_thread, stop_server)')

서버 도우미 준비 완료 (serve_in_thread, stop_server)


### 1.2 데이터셋: 캘리포니아 주택 가격

In [2]:
!pip install scikit-learn

In [3]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
print(f"샘플 수: {data.data.shape[0]:,}")
print(f"피처 수: {data.data.shape[1]}")
print(f"타겟: {data.target_names}")
print(f"\n피처 목록:")
for i, name in enumerate(data.feature_names):
    print(f"  {i+1}. {name}")

샘플 수: 20,640
피처 수: 8
타겟: ['MedHouseVal']

피처 목록:
  1. MedInc
  2. HouseAge
  3. AveRooms
  4. AveBedrms
  5. Population
  6. AveOccup
  7. Latitude
  8. Longitude


## 2. 모델 준비: 학습 → 전처리 파이프라인 → 저장

### 2.1 데이터 로드 및 탐색

In [4]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# 데이터 로드
data = fetch_california_housing()
X, y = data.data, data.target
feature_names = data.feature_names

print(f"피처 크기: {X.shape}")        # (20640, 8)
print(f"타겟 크기: {y.shape}")        # (20640,)
print(f"타겟 범위: {y.min():.2f} ~ {y.max():.2f} ($100,000 단위)")
print(f"\n피처별 통계:")
for i, name in enumerate(feature_names):
    print(f"  {name:12s}  평균: {X[:, i].mean():10.2f}  표준편차: {X[:, i].std():10.2f}")

피처 크기: (20640, 8)
타겟 크기: (20640,)
타겟 범위: 0.15 ~ 5.00 ($100,000 단위)

피처별 통계:
  MedInc        평균:       3.87  표준편차:       1.90
  HouseAge      평균:      28.64  표준편차:      12.59
  AveRooms      평균:       5.43  표준편차:       2.47
  AveBedrms     평균:       1.10  표준편차:       0.47
  Population    평균:    1425.48  표준편차:    1132.43
  AveOccup      평균:       3.07  표준편차:      10.39
  Latitude      평균:      35.63  표준편차:       2.14
  Longitude     평균:    -119.57  표준편차:       2.00


In [5]:
# 학습/테스트 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42  # *your code* — test_size와 random_state 설정
)

print(f"학습 데이터: {X_train.shape[0]:,}개")
print(f"테스트 데이터: {X_test.shape[0]:,}개")

학습 데이터: 16,512개
테스트 데이터: 4,128개


### 2.2 전처리: 정규화

In [6]:
# 학습 데이터의 평균/표준편차 계산 (테스트 데이터에는 학습 데이터의 통계를 사용)
train_mean = X_train.mean(axis=0)  # *your code* — 축(axis) 설정
train_std = X_train.std(axis=0)    # *your code* — 축(axis) 설정

print("피처별 평균:", np.round(train_mean, 2))
print("피처별 표준편차:", np.round(train_std, 2))

피처별 평균: [ 3.88000e+00  2.86100e+01  5.44000e+00  1.10000e+00  1.42645e+03
  3.10000e+00  3.56400e+01 -1.19580e+02]
피처별 표준편차: [1.90000e+00 1.26000e+01 2.39000e+00 4.30000e-01 1.13702e+03 1.15800e+01
 2.14000e+00 2.01000e+00]


In [7]:
# 정규화 적용
X_train_norm = (X_train - train_mean) / train_std  # *your code* — 정규화 공식
X_test_norm = (X_test - train_mean) / train_std     # 테스트에도 학습 데이터의 통계 사용

print(f"정규화 후 학습 데이터 평균: {X_train_norm.mean(axis=0).round(4)}")  # 거의 0
print(f"정규화 후 학습 데이터 표준편차: {X_train_norm.std(axis=0).round(4)}")  # 거의 1

정규화 후 학습 데이터 평균: [-0. -0.  0. -0. -0. -0.  0. -0.]
정규화 후 학습 데이터 표준편차: [1. 1. 1. 1. 1. 1. 1. 1.]


In [8]:
# 텐서 변환
X_train_tensor = torch.FloatTensor(X_train_norm)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)  # *your code* — (N,) → (N,1)
X_test_tensor = torch.FloatTensor(X_test_norm)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

print(f"X_train 텐서: {X_train_tensor.shape}")  # torch.Size([16512, 8])
print(f"y_train 텐서: {y_train_tensor.shape}")  # torch.Size([16512, 1])

X_train 텐서: torch.Size([16512, 8])
y_train 텐서: torch.Size([16512, 1])


### 2.3 모델 정의 및 학습

In [9]:
class HousingModel(nn.Module):
    """캘리포니아 주택 가격 예측 모델 (회귀)"""
    def __init__(self, input_dim=8):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),   # *your code* — 입력 차원 → 64
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),           # *your code* — 32 → 1 (회귀 출력)
        )

    def forward(self, x):
        return self.network(x)

model = HousingModel(input_dim=8)
print(f"모델 구조:\n{model}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

모델 구조:
HousingModel(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)
파라미터 수: 2,689


In [10]:
# 학습 설정
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

criterion = nn.MSELoss()                                          # *your code* — 회귀이므로 MSELoss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)         # *your code* — Adam 옵티마이저

EPOCHS = 50

In [11]:
# 학습 루프
model.train()
for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)          # *your code* — 순전파
        loss = criterion(predictions, y_batch) # *your code* — 손실 계산
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    if epoch % 10 == 0:
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch:3d}/{EPOCHS} — Loss: {avg_loss:.4f}")

Epoch  10/50 — Loss: 0.5383
Epoch  20/50 — Loss: 0.4771
Epoch  30/50 — Loss: 0.4425
Epoch  40/50 — Loss: 0.4100
Epoch  50/50 — Loss: 0.3865


In [12]:
# 테스트 평가
model.eval()
with torch.no_grad():
    test_preds = model(X_test_tensor)
    test_loss = criterion(test_preds, y_test_tensor)

    # MAE (Mean Absolute Error) 계산
    mae = torch.abs(test_preds - y_test_tensor).mean().item()

print(f"테스트 MSE:  {test_loss.item():.4f}")
print(f"테스트 MAE:  {mae:.4f} ($100,000 단위)")
print(f"테스트 MAE:  ${mae * 100000:,.0f} (실제 금액)")

테스트 MSE:  0.3266
테스트 MAE:  0.3868 ($100,000 단위)
테스트 MAE:  $38,681 (실제 금액)


### 2.4 모델 및 전처리 파라미터 저장

In [13]:
import os
os.makedirs("models", exist_ok=True)

# 모델 가중치 저장
torch.save(model.state_dict(), "models/housing_model.pth")  # *your code* — state_dict 저장
print(f"✅ 모델 저장: models/housing_model.pth ({os.path.getsize('models/housing_model.pth')/1024:.1f} KB)")

# 전처리 파라미터 저장 (배포 시 필수!)
preprocessing_params = {
    "mean": train_mean.tolist(),
    "std": train_std.tolist(),
    "feature_names": feature_names,
}

import json
with open("models/housing_preprocessing.json", "w") as f:
    json.dump(preprocessing_params, f, indent=2)

print(f"✅ 전처리 파라미터 저장: models/housing_preprocessing.json")

✅ 모델 저장: models/housing_model.pth (13.7 KB)
✅ 전처리 파라미터 저장: models/housing_preprocessing.json


### 2.5 추론 모듈 작성

In [14]:
%%writefile app/housing_model.py
"""
Day 5 - 주택 가격 예측 모델 정의 + 추론 함수
"""
import json
import torch
import torch.nn as nn
import numpy as np


class HousingModel(nn.Module):
    """캘리포니아 주택 가격 예측 모델"""
    def __init__(self, input_dim=8):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)


class HousingPredictor:
    """모델 로드 + 전처리 + 추론을 캡슐화한 클래스"""

    def __init__(self, model_path: str, preprocessing_path: str):
        # 전처리 파라미터 로드
        with open(preprocessing_path, "r") as f:
            params = json.load(f)
        self.mean = np.array(params["mean"])
        self.std = np.array(params["std"])
        self.feature_names = params["feature_names"]

        # 모델 로드
        self.model = HousingModel(input_dim=len(self.feature_names))
        self.model.load_state_dict(
            torch.load(model_path, map_location="cpu", weights_only=True)
        )
        self.model.eval()

    def predict(self, features: dict) -> dict:
        """
        피처 딕셔너리를 받아 가격을 예측합니다.

        Args:
            features: {"MedInc": 3.5, "HouseAge": 25, ...}
        Returns:
            {"predicted_price": 2.35, "predicted_price_usd": 235000}
        """
        # 피처를 올바른 순서로 배열
        values = [features[name] for name in self.feature_names]  # *your code* — 피처 순서 맞추기

        # 정규화
        values = np.array(values, dtype=np.float32)
        normalized = (values - self.mean) / self.std              # *your code* — 정규화 적용

        # 추론
        input_tensor = torch.FloatTensor(normalized).unsqueeze(0)  # (1, 8)
        with torch.no_grad():
            output = self.model(input_tensor)

        price = output.item()
        price = max(price, 0.0)  # 음수 방지

        return {
            "predicted_price": round(price, 4),
            "predicted_price_usd": int(price * 100000),
        }

Overwriting app/housing_model.py


In [15]:
# 모듈 테스트
import sys
sys.path.insert(0, ".")

from app.housing_model import HousingPredictor

predictor = HousingPredictor(
    model_path="models/housing_model.pth",
    preprocessing_path="models/housing_preprocessing.json",
)

# 테스트 데이터의 첫 번째 샘플로 테스트
sample_features = {name: float(X_test[0, i]) for i, name in enumerate(feature_names)}
print(f"입력 피처: {sample_features}")

result = predictor.predict(sample_features)
print(f"예측 가격: ${result['predicted_price_usd']:,}")
print(f"실제 가격: ${int(y_test[0] * 100000):,}")

입력 피처: {'MedInc': 1.6812, 'HouseAge': 25.0, 'AveRooms': 4.192200557103064, 'AveBedrms': 1.0222841225626742, 'Population': 1392.0, 'AveOccup': 3.8774373259052926, 'Latitude': 36.06, 'Longitude': -119.01}
예측 가격: $67,949
실제 가격: $47,700


## 3. FastAPI 백엔드: 추론 엔드포인트 + Pydantic 스키마

### 3.1 Pydantic 스키마 설계

In [16]:
%%writefile app/housing_schemas.py
"""
Day 5 - 주택 가격 예측 API 스키마
"""
from pydantic import BaseModel, Field


class HousingRequest(BaseModel):
    """주택 가격 예측 요청"""
    MedInc: float = Field(..., gt=0, description="중위 소득")                        # *your code* — gt=0 설정
    HouseAge: float = Field(..., ge=0, le=100, description="주택 연식 (년)")          # *your code* — ge, le 범위
    AveRooms: float = Field(..., gt=0, description="평균 방 수")
    AveBedrms: float = Field(..., gt=0, description="평균 침실 수")
    Population: float = Field(..., gt=0, description="인구")
    AveOccup: float = Field(..., gt=0, description="평균 거주 인원")
    Latitude: float = Field(..., ge=32, le=42, description="위도 (캘리포니아 범위)")   # *your code* — 캘리포니아 위도 범위
    Longitude: float = Field(..., ge=-125, le=-114, description="경도 (캘리포니아 범위)")

    model_config = {
        "json_schema_extra": {
            "examples": [
                {
                    "MedInc": 3.5,
                    "HouseAge": 25.0,
                    "AveRooms": 5.0,
                    "AveBedrms": 1.0,
                    "Population": 1500.0,
                    "AveOccup": 3.0,
                    "Latitude": 37.5,
                    "Longitude": -122.0,
                }
            ]
        }
    }


class HousingResponse(BaseModel):
    """주택 가격 예측 응답"""
    success: bool = Field(description="요청 처리 성공 여부")
    predicted_price: float = Field(description="예측 가격 ($100,000 단위)")
    predicted_price_usd: int = Field(description="예측 가격 (USD)")
    input_features: dict = Field(description="입력된 피처 값")

Overwriting app/housing_schemas.py


### 3.2 FastAPI 서버 구현

In [17]:
# Day 3에서 만든 로깅/에러 처리 파일을 확인하고, 없으면 새로 만듭니다.
# (아래에서 만들 housing_api.py 가 이 파일들을 import 합니다)
import os

os.makedirs("app", exist_ok=True)

BACKEND_FILES = {}

BACKEND_FILES["app/logger_config.py"] = r'''"""
Day 3 - 로깅 설정
"""
import logging
import sys


def setup_logger(name: str = "ml_api", level: str = "INFO") -> logging.Logger:
    """콘솔 로거를 설정합니다."""
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level))

    if logger.handlers:
        return logger

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.DEBUG)

    formatter = logging.Formatter(
        fmt="%(asctime)s %(levelname)-8s [%(name)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    return logger'''

BACKEND_FILES["app/error_handlers.py"] = r'''"""
Day 3 - 글로벌 에러 핸들러
"""
import traceback
import logging

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

logger = logging.getLogger("ml_api")


def register_error_handlers(app: FastAPI):
    """FastAPI 앱에 글로벌 에러 핸들러를 등록합니다."""

    @app.exception_handler(Exception)
    async def general_error_handler(request: Request, exc: Exception):
        """모든 예외를 잡아서 안전한 응답을 반환합니다."""
        logger.error(
            f"에러 발생: {type(exc).__name__}: {exc}\n"
            f"경로: {request.method} {request.url}\n"
            f"스택 트레이스:\n{traceback.format_exc()}"
        )
        return JSONResponse(
            status_code=500,
            content={
                "success": False,
                "error": "서버 내부 오류가 발생했습니다.",
            }
            # ⚠️ 클라이언트에게는 상세 정보를 노출하지 않습니다.
            # 상세 정보는 서버 로그에만 기록됩니다.
        )'''

BACKEND_FILES["app/middleware.py"] = r'''"""
Day 3 - 요청/응답 로깅 미들웨어
모든 요청의 메서드, 경로, 응답 시간, 상태 코드를 자동 로깅합니다.
"""
import time
import logging
from fastapi import Request
from starlette.middleware.base import BaseHTTPMiddleware

logger = logging.getLogger("ml_api")


class RequestLoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        start_time = time.time()
        response = await call_next(request)
        duration = round(time.time() - start_time, 3)

        log_message = (
            f"{request.method} {request.url.path} "
            f"→ {response.status_code} "
            f"({duration}s)"
        )

        if response.status_code >= 500:
            logger.error(log_message)
        elif response.status_code >= 400:
            logger.warning(log_message)
        else:
            logger.info(log_message)

        response.headers["X-Process-Time"] = str(duration)
        return response'''

for _path, _body in BACKEND_FILES.items():
    if os.path.exists(_path):
        print(f"✅ {_path} 있음")
    else:
        print(f"⚠️ {_path} 없음 → 생성합니다.")
        with open(_path, "w", encoding="utf-8") as f:
            f.write(_body)

print("\n🎉 준비 완료. 다음 셀로 진행하세요.")


✅ app/logger_config.py 있음
✅ app/error_handlers.py 있음
✅ app/middleware.py 있음

🎉 준비 완료. 다음 셀로 진행하세요.


In [18]:
%%writefile app/housing_api.py
"""
Day 5 - 주택 가격 예측 FastAPI 서버
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, HTTPException

from app.housing_schemas import HousingRequest, HousingResponse
from app.housing_model import HousingPredictor
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware


# ===== 설정 =====
logger = setup_logger("housing_api")

app = FastAPI(
    title="California Housing Price API",
    description="캘리포니아 주택 가격을 예측하는 API",
    version="1.0.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

# 추론 전용 스레드풀 (Day 3에서 배운 패턴)
inference_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="housing")  # *your code* — 스레드풀 생성

# ===== 모델 로드 =====
MODEL_PATH = "models/housing_model.pth"
PREPROCESS_PATH = "models/housing_preprocessing.json"
predictor = None


@app.on_event("startup")
async def startup():
    global predictor
    logger.info("주택 가격 모델 로드 중...")
    predictor = HousingPredictor(MODEL_PATH, PREPROCESS_PATH)  # *your code* — HousingPredictor 인스턴스 생성
    logger.info("모델 로드 완료")


# ===== 엔드포인트 =====

@app.get("/health", tags=["System"])
async def health_check():
    return {
        "status": "healthy" if predictor is not None else "loading",
        "model": "California Housing",
    }


@app.post("/predict", response_model=HousingResponse, tags=["Prediction"])
async def predict_housing(request: HousingRequest):
    """주택 정보를 받아 가격을 예측합니다."""
    if predictor is None:
        raise HTTPException(status_code=503, detail="모델이 아직 로드되지 않았습니다.")

    # 요청 데이터를 딕셔너리로 변환
    features = request.model_dump()  # *your code* — Pydantic 모델 → dict

    try:
        # 추론 (별도 스레드에서 실행 — Day 3 패턴)
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(       # *your code* — run_in_executor 사용
            inference_executor,
            predictor.predict,
            features,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    return HousingResponse(
        success=True,
        predicted_price=result["predicted_price"],
        predicted_price_usd=result["predicted_price_usd"],
        input_features=features,
    )

Overwriting app/housing_api.py


### 3.3 서버 실행 및 Swagger 테스트

In [19]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
serve_in_thread("app.housing_api:app", port=8000)

2026-08-19 14:39:52 INFO     [housing_api] 주택 가격 모델 로드 중...
2026-08-19 14:39:52 INFO     [housing_api] 모델 로드 완료
서버 실행됨: http://127.0.0.1:8000


In [20]:
# Swagger UI를 노트북 안에서 열기
try:
    from google.colab import output as _colab_output   # Colab이면 import에 성공합니다
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import output
    print("✅ Swagger UI를 아래에 띄웁니다 (Colab 프록시 → 포트 8000)")
    output.serve_kernel_port_as_iframe(8000, path="/docs", height="700")
else:
    from IPython.display import IFrame, display
    print("✅ Swagger UI를 아래에 띄웁니다: http://localhost:8000/docs")
    display(IFrame("http://localhost:8000/docs", width="100%", height=700))

✅ Swagger UI를 아래에 띄웁니다: http://localhost:8000/docs


In [21]:
import requests, json

# 헬스체크
resp = requests.get("http://localhost:8000/health")
print(f"헬스체크: {resp.json()}")

헬스체크: {'status': 'healthy', 'model': 'California Housing'}


In [22]:
# 추론 테스트
sample_request = {
    "MedInc": 3.5,
    "HouseAge": 25.0,
    "AveRooms": 5.0,
    "AveBedrms": 1.0,
    "Population": 1500.0,
    "AveOccup": 3.0,
    "Latitude": 37.5,
    "Longitude": -122.0,
}

resp = requests.post("http://localhost:8000/predict", json=sample_request)
result = resp.json()

print(f"상태 코드: {resp.status_code}")
print(f"예측 가격: ${result['predicted_price_usd']:,}")
print(f"전체 응답:")
print(json.dumps(result, indent=2, ensure_ascii=False))

상태 코드: 200
예측 가격: $183,971
전체 응답:
{
  "success": true,
  "predicted_price": 1.8397,
  "predicted_price_usd": 183971,
  "input_features": {
    "MedInc": 3.5,
    "HouseAge": 25.0,
    "AveRooms": 5.0,
    "AveBedrms": 1.0,
    "Population": 1500.0,
    "AveOccup": 3.0,
    "Latitude": 37.5,
    "Longitude": -122.0
  }
}


In [23]:
# 에러 테스트: 필수 필드 누락
resp = requests.post("http://localhost:8000/predict", json={"MedInc": 3.5})
print(f"필드 누락 → 상태: {resp.status_code}")

# 에러 테스트: 범위 초과
resp = requests.post("http://localhost:8000/predict", json={
    **sample_request, "Latitude": 50.0  # 캘리포니아 범위 초과
})
print(f"범위 초과 → 상태: {resp.status_code}")

POST /predict → 422 (0.0s)


필드 누락 → 상태: 422


POST /predict → 422 (0.0s)


범위 초과 → 상태: 422


## 4. Streamlit 프론트엔드: 입력 폼 → API 호출 → 결과 시각화

### 4.1 대시보드 코드 작성

In [24]:
%%writefile frontend/app_housing.py
"""
Day 5 - 캘리포니아 주택 가격 예측 대시보드
"""
import streamlit as st
import requests

# ===== 페이지 설정 =====
st.set_page_config(
    page_title="주택 가격 예측",
    page_icon="🏠",
    layout="wide",
)

# ===== API 호출 함수 (Day 4 패턴) =====
API_BASE = "http://localhost:8000"

def call_api(url, json_data=None, method="post"):
    try:
        if method == "get":
            resp = requests.get(url, timeout=10)
        else:
            resp = requests.post(url, json=json_data, timeout=30)
        resp.raise_for_status()                      # *your code* — HTTP 에러 시 예외 발생
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error("🔌 **서버에 연결할 수 없습니다.** FastAPI 서버를 실행하세요.")
        return None
    except requests.exceptions.HTTPError as e:
        st.error(f"❌ **서버 에러** (HTTP {e.response.status_code})")
        return None
    except Exception as e:
        st.error(f"❌ **오류:** {type(e).__name__}")
        return None


# ===== 사이드바 =====
with st.sidebar:
    st.header("⚙️ 설정")

    health = call_api(f"{API_BASE}/health", method="get")
    if health and health.get("status") == "healthy":
        st.success("🟢 서버 연결됨")
        server_ok = True
    else:
        st.error("🔴 서버 연결 실패")
        server_ok = False

    st.divider()
    st.caption("California Housing Price Predictor")
    st.caption("Day 5 — 프로젝트 1")


# ===== 메인 영역 =====
st.title("🏠 캘리포니아 주택 가격 예측")
st.write("주택 정보를 입력하면 예상 가격을 예측합니다.")

col_input, col_result = st.columns(2)

# ----- 입력 영역 -----
with col_input:
    st.subheader("📋 주택 정보 입력")

    # 소득 & 주택 연식
    c1, c2 = st.columns(2)
    with c1:
        med_inc = st.number_input(
            "중위 소득 (MedInc)",
            min_value=0.1, max_value=20.0, value=3.5, step=0.1,   # *your code* — 범위와 기본값
        )
    with c2:
        house_age = st.number_input(
            "주택 연식 (HouseAge)",
            min_value=0.0, max_value=100.0, value=25.0, step=1.0,
        )

    # 방 수 & 침실 수
    c1, c2 = st.columns(2)
    with c1:
        ave_rooms = st.number_input(
            "평균 방 수 (AveRooms)",
            min_value=0.1, max_value=50.0, value=5.0, step=0.1,
        )
    with c2:
        ave_bedrms = st.number_input(
            "평균 침실 수 (AveBedrms)",
            min_value=0.1, max_value=20.0, value=1.0, step=0.1,
        )

    # 인구 & 거주 인원
    c1, c2 = st.columns(2)
    with c1:
        population = st.number_input(
            "인구 (Population)",
            min_value=1.0, max_value=50000.0, value=1500.0, step=100.0,
        )
    with c2:
        ave_occup = st.number_input(
            "평균 거주 인원 (AveOccup)",
            min_value=0.1, max_value=20.0, value=3.0, step=0.1,
        )

    # 위치
    c1, c2 = st.columns(2)
    with c1:
        latitude = st.number_input(
            "위도 (Latitude)",
            min_value=32.0, max_value=42.0, value=37.5, step=0.1,   # *your code* — 캘리포니아 범위
        )
    with c2:
        longitude = st.number_input(
            "경도 (Longitude)",
            min_value=-125.0, max_value=-114.0, value=-122.0, step=0.1,
        )


# ----- 결과 영역 -----
with col_result:
    st.subheader("📊 예측 결과")

    if not server_ok:
        st.error("서버에 연결할 수 없습니다.")
    else:
        if st.button("🚀 가격 예측", type="primary", use_container_width=True):
            # 요청 데이터 구성
            request_data = {                            # *your code* — 입력값을 dict로 구성
                "MedInc": med_inc,
                "HouseAge": house_age,
                "AveRooms": ave_rooms,
                "AveBedrms": ave_bedrms,
                "Population": population,
                "AveOccup": ave_occup,
                "Latitude": latitude,
                "Longitude": longitude,
            }

            with st.spinner("예측 중..."):
                result = call_api(f"{API_BASE}/predict", json_data=request_data)

            if result:
                st.session_state["last_housing_result"] = result

        # 결과 표시
        if "last_housing_result" in st.session_state:
            result = st.session_state["last_housing_result"]

            # 가격 메트릭
            st.metric(
                label="예상 주택 가격",
                value=f"${result['predicted_price_usd']:,}",
            )

            st.caption(f"모델 출력값: {result['predicted_price']} ($100,000 단위)")

            # 입력 피처 확인
            with st.expander("📋 입력된 피처 확인"):
                for key, value in result["input_features"].items():
                    st.write(f"**{key}**: {value}")

Overwriting frontend/app_housing.py


### 4.2 실행 및 테스트

In [25]:
# Colab은 세션이 새로 뜰 때마다 streamlit 설치가 필요합니다.
try:
    import streamlit
except ImportError:
    print("❌ Streamlit 미설치 → 지금 설치합니다. (1분 정도 걸립니다)")
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"], check=True)

import sys, subprocess, time, socket, contextlib, tempfile, os

try:
    from google.colab import output as _colab_output   # Colab이면 import에 성공합니다
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def run_streamlit(script, port=8501):
    """Streamlit을 백그라운드로 띄우고 '실제로 떴는지'까지 확인한다. (Windows/macOS/Linux 공통)"""
    def port_open(p):
        with contextlib.closing(socket.socket()) as s:
            s.settimeout(0.5)
            return s.connect_ex(("127.0.0.1", p)) == 0

    if port_open(port):                      # 이미 떠 있으면 재사용
        print(f"♻️  이미 실행 중 (포트 {port})")
        return None

    # 로그는 파일로 — 파이프가 가득 차 서버가 멈추는 일을 막고, 실패 시 원인을 읽을 수 있다
    log_path = os.path.join(tempfile.gettempdir(), f"streamlit_{port}.log")
    log = open(log_path, "w", encoding="utf-8")

    proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", script,
         "--server.port", str(port),
         "--server.enableCORS", "false",           # iframe은 '다른 주소'로 취급됩니다 —
         "--server.enableXsrfProtection", "false", #   끄지 않으면 화면이 계속 로딩만 됩니다
         "--server.headless", "true"],             # 최초 실행 '이메일 프롬프트'를 건너뜀
        stdout=log, stderr=subprocess.STDOUT,      #   (없으면 입력을 기다리다 조용히 죽음)
    )
    for _ in range(60):                      # 최대 15초, 0.25초 간격 확인
        if proc.poll() is not None:          # 일찍 죽음 → 로그를 보여줌
            log.close()
            print(f"❌ Streamlit이 종료됨 (code {proc.returncode}) — 로그:")
            print(open(log_path, encoding="utf-8").read()[-2000:])
            return proc
        if port_open(port):
            if IN_COLAB:                     # Colab의 localhost는 '내 PC'가 아니라 Colab 서버입니다
                print(f"✅ 프론트엔드 실행됨 (포트 {port}) — 아래 Step 3에서 화면을 띄웁니다")
            else:
                print(f"✅ 프론트엔드: http://localhost:{port}")
            print(f"   (로그: {log_path})")
            return proc
        time.sleep(0.25)
    proc.terminate(); log.close()
    print(f"❌ 15초 내에 포트가 열리지 않음 — 로그:")
    print(open(log_path, encoding="utf-8").read()[-2000:])
    return proc

proc = run_streamlit("frontend/app_housing.py", port=8501)



✅ 프론트엔드: http://localhost:8501
   (로그: C:\Users\하노\AppData\Local\Temp\streamlit_8501.log)


In [26]:
def show_dashboard(port=8501, height=900):
    """실행 중인 프론트엔드를 노트북 셀 안에 iframe으로 띄웁니다."""
    def port_open(p):
        with contextlib.closing(socket.socket()) as s:
            s.settimeout(0.5)
            return s.connect_ex(("127.0.0.1", p)) == 0

    if not port_open(port):                  # Step 2를 건너뛴 경우
        print(f"⚠️ 포트 {port}에 프론트엔드가 없습니다. Step 2 셀을 먼저 실행하세요.")
        return

    if IN_COLAB:
        from google.colab import output
        print(f"✅ 대시보드를 아래에 띄웁니다 (Colab 프록시 → 포트 {port})")
        output.serve_kernel_port_as_iframe(port, height=str(height))
    else:
        from IPython.display import IFrame, display
        print(f"✅ 대시보드를 아래에 띄웁니다: http://localhost:{port}")
        display(IFrame(f"http://localhost:{port}", width="100%", height=height))

show_dashboard(port=8501, height=900)


✅ 대시보드를 아래에 띄웁니다: http://localhost:8501


## 5. 통합 테스트 및 디버깅

### 5.1 API 레벨 통합 테스트

In [27]:
# ⚠️ FastAPI 서버가 실행 중이어야 합니다.

import requests
import json
import time

API_BASE = "http://localhost:8000"

test_results = {}   # 각 테스트가 실제 결과를 여기에 기록합니다 — 마지막 종합에서 사용

print("=" * 60)
print("  통합 테스트")
print("=" * 60)


  통합 테스트


### 테스트 1: 정상 요청

In [28]:
# 다양한 입력으로 모델 동작 확인
test_cases = [
    {"name": "저소득 지역", "MedInc": 1.5, "HouseAge": 40, "AveRooms": 4.0, "AveBedrms": 1.0,
     "Population": 2000, "AveOccup": 3.5, "Latitude": 34.0, "Longitude": -118.0},
    {"name": "고소득 지역", "MedInc": 10.0, "HouseAge": 10, "AveRooms": 8.0, "AveBedrms": 2.0,
     "Population": 500, "AveOccup": 2.0, "Latitude": 37.8, "Longitude": -122.4},
    {"name": "평균적 주택", "MedInc": 3.5, "HouseAge": 25, "AveRooms": 5.0, "AveBedrms": 1.0,
     "Population": 1500, "AveOccup": 3.0, "Latitude": 37.5, "Longitude": -122.0},
]

print("\n[테스트 1] 정상 요청 — 다양한 입력")
print(f"{'케이스':<15} {'예측 가격':>12}")
print("-" * 30)

statuses = []
for case in test_cases:
    name = case.pop("name")
    resp = requests.post(f"{API_BASE}/predict", json=case)    # *your code* — POST 요청
    statuses.append(resp.status_code)
    if resp.status_code == 200:
        print(f"{name:<15} ${resp.json()['predicted_price_usd']:>10,}")
    else:
        print(f"{name:<15} ❌ HTTP {resp.status_code}")
    case["name"] = name  # 복원

test_results["정상 요청"] = all(s == 200 for s in statuses)   # 세 케이스 모두 200이어야 통과



[테스트 1] 정상 요청 — 다양한 입력
케이스                    예측 가격
------------------------------
저소득 지역          $    99,792
고소득 지역          $   503,372
평균적 주택          $   183,971


### 테스트 2: 에러 상황

In [29]:
print("\n[테스트 2] 에러 상황")
error_statuses = []

# 필수 필드 누락
resp = requests.post(f"{API_BASE}/predict", json={"MedInc": 3.5})
error_statuses.append(resp.status_code)
print(f"  필드 누락      → HTTP {resp.status_code}")

# 범위 초과 (위도)
bad_request = {
    "MedInc": 3.5, "HouseAge": 25, "AveRooms": 5, "AveBedrms": 1,
    "Population": 1500, "AveOccup": 3, "Latitude": 50.0, "Longitude": -122.0,  # 위도 초과
}
resp = requests.post(f"{API_BASE}/predict", json=bad_request)
error_statuses.append(resp.status_code)
print(f"  위도 범위 초과  → HTTP {resp.status_code}")

# 음수 값
bad_request2 = {
    "MedInc": -1.0, "HouseAge": 25, "AveRooms": 5, "AveBedrms": 1,
    "Population": 1500, "AveOccup": 3, "Latitude": 37.5, "Longitude": -122.0,
}
resp = requests.post(f"{API_BASE}/predict", json=bad_request2)
error_statuses.append(resp.status_code)
print(f"  소득 음수      → HTTP {resp.status_code}")

# JSON이 아닌 요청
resp = requests.post(f"{API_BASE}/predict", data="not json")
error_statuses.append(resp.status_code)
print(f"  잘못된 포맷    → HTTP {resp.status_code}")

# 모두 4xx(클라이언트 잘못)로 거부되어야 통과 — 200(통과시킴)이나 500(서버 죽음)이면 실패
test_results["에러 처리"] = all(400 <= s < 500 for s in error_statuses)



[테스트 2] 에러 상황


POST /predict → 422 (0.0s)


  필드 누락      → HTTP 422


POST /predict → 422 (0.0s)


  위도 범위 초과  → HTTP 422


POST /predict → 422 (0.0s)


  소득 음수      → HTTP 422


POST /predict → 422 (0.0s)


  잘못된 포맷    → HTTP 422


### 테스트 3: 동시 요청

In [30]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def send_predict(i):
    case = test_cases[i % len(test_cases)].copy()
    case.pop("name", None)
    start = time.time()
    resp = requests.post(f"{API_BASE}/predict", json=case, timeout=30)
    return {"id": i+1, "elapsed": round(time.time() - start, 3), "status": resp.status_code}

print("\n[테스트 3] 동시 요청 (8개)")
start = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(send_predict, i) for i in range(8)]
    results = [f.result() for f in as_completed(futures)]

total = round(time.time() - start, 2)
for r in sorted(results, key=lambda x: x["id"]):
    print(f"  요청 #{r['id']}: {r['elapsed']}초 (HTTP {r['status']})")
print(f"  전체: {total}초")

test_results["동시 처리"] = all(r["status"] == 200 for r in results)   # 8건 모두 200이어야 통과


[테스트 3] 동시 요청 (8개)
  요청 #1: 2.041초 (HTTP 200)
  요청 #2: 2.041초 (HTTP 200)
  요청 #3: 2.195초 (HTTP 200)
  요청 #4: 2.195초 (HTTP 200)
  요청 #5: 2.036초 (HTTP 200)
  요청 #6: 2.19초 (HTTP 200)
  요청 #7: 2.194초 (HTTP 200)
  요청 #8: 2.189초 (HTTP 200)
  전체: 2.2초


### 테스트 4: 헬스체크

In [31]:
print("\n[테스트 4] 헬스체크")
resp = requests.get(f"{API_BASE}/health")
print(f"  상태: {resp.json()}")

test_results["헬스체크"] = resp.status_code == 200 and resp.json().get("status") == "healthy"



[테스트 4] 헬스체크
  상태: {'status': 'healthy', 'model': 'California Housing'}


### 5.2 테스트 결과 종합

In [32]:
print("\n" + "=" * 60)
print("  테스트 결과 종합")
print("=" * 60)

# 위 테스트 셀들이 test_results에 기록한 '실제 결과'로 판정합니다.
DESCRIPTIONS = {
    "정상 요청": "다양한 입력에서 200과 예측 가격 반환",
    "에러 처리": "잘못된 입력을 4xx로 거부, 서버 안 죽음",
    "동시 처리": "8개 동시 요청 모두 200",
    "헬스체크": "서버 상태 healthy",
}

for key, desc in DESCRIPTIONS.items():
    if key not in test_results:
        print(f"  ⚠️ {key}: 아직 실행되지 않음 — 위 테스트 셀을 먼저 실행하세요")
    elif test_results[key]:
        print(f"  ✅ {key}: {desc}")
    else:
        print(f"  ❌ {key}: 실패 — 위 테스트 셀의 출력을 확인하세요")

if all(test_results.get(k) for k in DESCRIPTIONS):
    print("\n🎉 4가지 테스트를 모두 통과했습니다. 회고를 작성하고 제출하세요!")
else:
    print("\n⚠️ 통과하지 못한 테스트가 있습니다. 서버 상태와 위 셀들의 출력을 확인하세요.")


  테스트 결과 종합
  ✅ 정상 요청: 다양한 입력에서 200과 예측 가격 반환
  ✅ 에러 처리: 잘못된 입력을 4xx로 거부, 서버 안 죽음
  ✅ 동시 처리: 8개 동시 요청 모두 200
  ✅ 헬스체크: 서버 상태 healthy

🎉 4가지 테스트를 모두 통과했습니다. 회고를 작성하고 제출하세요!
